# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_13 — Order Book Reconstruction from Events

### Research question

How can we reconstruct a limit order book from raw event messages, validate sequence integrity, repair from snapshots, and produce trustworthy L1/L2 microstructure features for execution research?

This notebook closes **Phase 6: Execution, Market Microstructure & Systems**.

It follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
06_06_microstructure_friction_cpp_core
06_07_limit_board_margin_and_deadband_rebalancing
06_08_avellaneda_stoikov_market_making
06_09_limit_order_fill_probability_model
06_10_smart_order_router_simulator
06_11_cpp_python_bindings_pybind11
06_12_latency_budget_profiler
06_13_order_book_reconstruction_from_events
```

The previous notebooks built execution models, fill models, routing models, bindings, and latency profiling. This notebook focuses on the foundation underneath all of them:

> reconstructing the state of the order book from raw event data.

It covers:

1. event schemas;
2. snapshots versus incremental updates;
3. sequence numbers;
4. out-of-order events;
5. dropped event detection;
6. duplicate event detection;
7. add/modify/cancel/trade messages;
8. price-level book state;
9. order-level versus price-level reconstruction;
10. L1 best bid/ask extraction;
11. L2 depth features;
12. order-flow imbalance;
13. microprice;
14. book pressure;
15. trade-through checks;
16. crossed-book checks;
17. checksum validation;
18. gap recovery from snapshots;
19. stale quote detection;
20. replay speed benchmarks;
21. reconstructed feature tables;
22. reconciliation reports;
23. governance flags;
24. audit checks;
25. saved outputs and manifest.

The key idea:

> A microstructure model is only as trustworthy as the reconstructed book it is trained on.

## 1. Why order book reconstruction matters

Many execution and market microstructure models need top-of-book or book-depth state:

- fill probability models;
- smart order routers;
- market-making models;
- microprice signals;
- L1/L2 execution backtests;
- queue-position models;
- toxicity/adverse-selection models.

But raw feeds often arrive as event messages:

```text
snapshot
add
modify
cancel
trade
delete
heartbeat
```

To use those messages, we need to replay them into book state.

The reconstruction must answer:

1. Did we process every event in sequence?
2. Did duplicate events appear?
3. Did we lose messages?
4. Did snapshots and increments agree?
5. Did the book ever cross?
6. Did trades happen through the book?
7. Are best bid and ask trustworthy?
8. Can this data be used for execution research?

Bad reconstruction quietly contaminates every downstream model.

## 2. Snapshot and incremental feeds

A typical market-data feed provides:

### Snapshot

Full or partial state at a point in time:

$$
Book_t = \{(bidPrice_i,bidSize_i),(askPrice_i,askSize_i)\}_{i=1}^{K}
$$

### Incremental update

A change after the snapshot:

- add liquidity;
- modify size;
- cancel liquidity;
- execute trade;
- delete price level.

A robust process is:

```text
load snapshot
check sequence
apply increments
detect gaps
recover from next snapshot
validate checksum
emit features
```

## 3. Price-level versus order-level reconstruction

### Order-level reconstruction

Tracks every individual order ID.

Pros:

- true queue position possible;
- accurate partial cancellations;
- order age features.

Cons:

- large memory footprint;
- requires order IDs;
- harder to replay.

### Price-level reconstruction

Tracks aggregate size at each price level.

Pros:

- lighter;
- enough for L1/L2 features;
- common in aggregated feeds.

Cons:

- cannot recover true queue position;
- cannot identify individual order priority.

This notebook implements **price-level reconstruction**, while simulating event messages that include synthetic order IDs for diagnostics.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class OrderBookReconstructionConfig:
    n_events: int = 90_000
    n_levels: int = 10
    seed: int = 42
    initial_mid: float = 100.0
    tick_size: float = 0.01
    base_level_size: float = 2_500.0
    snapshot_interval: int = 7_500
    corruption_duplicate_prob: float = 0.0008
    corruption_drop_prob: float = 0.0008
    corruption_shuffle_window: int = 5
    max_recovery_gap: int = 50
    stale_quote_event_threshold: int = 2_000
    benchmark_repeats: int = 3
    output_subdir: str = "order_book_reconstruction_from_events_v1"

config = OrderBookReconstructionConfig()
config

## 5. Book representation helpers

The book is represented as two dictionaries:

```python
bids: price -> size
asks: price -> size
```

Bid prices should be below ask prices.

Best bid:

$$
\max(price \in bids)
$$

Best ask:

$$
\min(price \in asks)
$$

In [ ]:
def round_to_tick(price, tick):
    return float(np.round(price / tick) * tick)

def initialise_book(mid, n_levels, tick_size, base_size, rng):
    mid = round_to_tick(mid, tick_size)

    bids = {}
    asks = {}

    for level in range(1, n_levels + 1):
        bid_price = round_to_tick(mid - level * tick_size, tick_size)
        ask_price = round_to_tick(mid + level * tick_size, tick_size)

        size_bid = base_size * rng.lognormal(0.0, 0.25) / np.sqrt(level)
        size_ask = base_size * rng.lognormal(0.0, 0.25) / np.sqrt(level)

        bids[bid_price] = float(size_bid)
        asks[ask_price] = float(size_ask)

    return bids, asks

def clean_book_side(side_dict):
    return {float(p): float(s) for p, s in side_dict.items() if s > 1e-12}

def best_bid_ask(bids, asks):
    bids = clean_book_side(bids)
    asks = clean_book_side(asks)

    best_bid = max(bids.keys()) if bids else np.nan
    best_ask = min(asks.keys()) if asks else np.nan
    bid_size = bids.get(best_bid, np.nan) if bids else np.nan
    ask_size = asks.get(best_ask, np.nan) if asks else np.nan

    return best_bid, best_ask, bid_size, ask_size

def book_to_levels(bids, asks, n_levels):
    bid_items = sorted(clean_book_side(bids).items(), key=lambda x: x[0], reverse=True)[:n_levels]
    ask_items = sorted(clean_book_side(asks).items(), key=lambda x: x[0])[:n_levels]

    row = {}
    for i in range(n_levels):
        if i < len(bid_items):
            row[f"bid_px_{i+1}"] = bid_items[i][0]
            row[f"bid_sz_{i+1}"] = bid_items[i][1]
        else:
            row[f"bid_px_{i+1}"] = np.nan
            row[f"bid_sz_{i+1}"] = 0.0

        if i < len(ask_items):
            row[f"ask_px_{i+1}"] = ask_items[i][0]
            row[f"ask_sz_{i+1}"] = ask_items[i][1]
        else:
            row[f"ask_px_{i+1}"] = np.nan
            row[f"ask_sz_{i+1}"] = 0.0

    return row

def simple_book_checksum(bids, asks, n_levels=5):
    levels = book_to_levels(bids, asks, n_levels)
    value = 0
    for k, v in levels.items():
        if pd.isna(v):
            x = 0
        else:
            x = int(round(float(v) * 100))
        value = (value * 1315423911 + x) % 2_147_483_647
    return int(value)

rng = np.random.default_rng(config.seed)
bids0, asks0 = initialise_book(config.initial_mid, config.n_levels, config.tick_size, config.base_level_size, rng)

best_bid_ask(bids0, asks0), simple_book_checksum(bids0, asks0)

## 6. Apply event to book

Supported event types:

| Event | Meaning |
|---|---|
| `SNAPSHOT` | reset book state |
| `ADD` | add size at price |
| `MODIFY` | set size at price |
| `CANCEL` | reduce size at price |
| `TRADE` | remove size from opposite side |
| `DELETE` | remove price level |

For price-level reconstruction, `ADD` and `CANCEL` are aggregate changes, not individual order operations.

In [ ]:
def apply_event_to_book(bids, asks, event):
    event_type = event["event_type"]

    if event_type == "SNAPSHOT":
        return dict(event["snapshot_bids"]), dict(event["snapshot_asks"])

    side = event["side"]
    price = float(event["price"])
    size = float(event["size"])

    side_dict = bids if side == "BID" else asks

    if event_type == "ADD":
        side_dict[price] = side_dict.get(price, 0.0) + size

    elif event_type == "MODIFY":
        if size <= 0:
            side_dict.pop(price, None)
        else:
            side_dict[price] = size

    elif event_type == "CANCEL":
        current = side_dict.get(price, 0.0)
        new_size = max(0.0, current - size)
        if new_size <= 1e-12:
            side_dict.pop(price, None)
        else:
            side_dict[price] = new_size

    elif event_type == "DELETE":
        side_dict.pop(price, None)

    elif event_type == "TRADE":
        # Trade removes liquidity from the resting side specified in event["side"].
        current = side_dict.get(price, 0.0)
        new_size = max(0.0, current - size)
        if new_size <= 1e-12:
            side_dict.pop(price, None)
        else:
            side_dict[price] = new_size

    else:
        raise ValueError(f"Unknown event_type: {event_type}")

    return bids, asks

## 7. Simulate clean exchange event stream

We generate a clean event stream with sequence numbers.

The simulation maintains a hidden true book and emits incremental messages.

Every `snapshot_interval` events, a full snapshot is emitted for recovery.

In [ ]:
def random_existing_level(side_dict, rng):
    if not side_dict:
        return None
    prices = np.array(list(side_dict.keys()), dtype=float)
    sizes = np.array([side_dict[p] for p in prices], dtype=float)
    weights = sizes / sizes.sum() if sizes.sum() > 0 else np.ones(len(prices)) / len(prices)
    price = float(rng.choice(prices, p=weights))
    return price

def simulate_clean_event_stream(config):
    rng = np.random.default_rng(config.seed + 100)
    bids, asks = initialise_book(config.initial_mid, config.n_levels, config.tick_size, config.base_level_size, rng)

    rows = []
    snapshots = {}
    order_id_counter = 1
    event_time_ns = 0

    for seq in range(1, config.n_events + 1):
        event_time_ns += int(rng.exponential(50_000))

        if seq == 1 or seq % config.snapshot_interval == 0:
            checksum = simple_book_checksum(bids, asks, n_levels=5)
            snapshot_bids = dict(bids)
            snapshot_asks = dict(asks)
            snapshots[seq] = {
                "snapshot_bids": snapshot_bids,
                "snapshot_asks": snapshot_asks,
                "checksum": checksum,
            }
            rows.append({
                "seq": seq,
                "event_time_ns": event_time_ns,
                "event_type": "SNAPSHOT",
                "side": "",
                "price": np.nan,
                "size": np.nan,
                "order_id": -1,
                "aggressor_side": "",
                "checksum": checksum,
                "snapshot_bids": snapshot_bids,
                "snapshot_asks": snapshot_asks,
            })
            continue

        best_bid, best_ask, _, _ = best_bid_ask(bids, asks)
        mid = (best_bid + best_ask) / 2.0

        event_type = rng.choice(
            ["ADD", "MODIFY", "CANCEL", "TRADE", "DELETE"],
            p=[0.42, 0.18, 0.20, 0.17, 0.03],
        )

        if event_type == "ADD":
            side = rng.choice(["BID", "ASK"])
            level = int(rng.integers(1, config.n_levels + 5))
            if side == "BID":
                price = round_to_tick(best_bid - rng.integers(0, 3) * config.tick_size if rng.uniform() < 0.65 else mid - level * config.tick_size, config.tick_size)
                price = min(price, best_ask - config.tick_size)
            else:
                price = round_to_tick(best_ask + rng.integers(0, 3) * config.tick_size if rng.uniform() < 0.65 else mid + level * config.tick_size, config.tick_size)
                price = max(price, best_bid + config.tick_size)
            size = float(rng.lognormal(np.log(config.base_level_size / 3), 0.65))
            order_id = order_id_counter
            order_id_counter += 1

        elif event_type in ["MODIFY", "CANCEL", "DELETE"]:
            side = rng.choice(["BID", "ASK"])
            side_dict = bids if side == "BID" else asks
            price = random_existing_level(side_dict, rng)
            if price is None:
                event_type = "ADD"
                side = rng.choice(["BID", "ASK"])
                price = best_bid if side == "BID" else best_ask
                size = float(rng.lognormal(np.log(config.base_level_size / 3), 0.65))
            else:
                current = side_dict.get(price, 0.0)
                if event_type == "MODIFY":
                    size = float(max(1.0, current * rng.lognormal(0.0, 0.35)))
                elif event_type == "CANCEL":
                    size = float(current * rng.uniform(0.05, 0.80))
                else:
                    size = float(current)
            order_id = int(rng.integers(1, max(order_id_counter, 2)))

        else:
            # Trade removes resting liquidity at the touch or near touch.
            aggressor_side = rng.choice(["BUY", "SELL"])
            if aggressor_side == "BUY":
                side = "ASK"
                price = best_ask
                current = asks.get(price, config.base_level_size)
            else:
                side = "BID"
                price = best_bid
                current = bids.get(price, config.base_level_size)

            size = float(min(current, rng.lognormal(np.log(config.base_level_size / 5), 0.75)))
            order_id = -1

        event = {
            "event_type": event_type,
            "side": side,
            "price": price,
            "size": size,
        }

        bids, asks = apply_event_to_book(bids, asks, event)

        # Replenish if one side becomes empty.
        if not bids or not asks:
            bids, asks = initialise_book(mid, config.n_levels, config.tick_size, config.base_level_size, rng)

        checksum = simple_book_checksum(bids, asks, n_levels=5)

        rows.append({
            "seq": seq,
            "event_time_ns": event_time_ns,
            "event_type": event_type,
            "side": side,
            "price": price,
            "size": size,
            "order_id": order_id,
            "aggressor_side": aggressor_side if event_type == "TRADE" else "",
            "checksum": checksum,
            "snapshot_bids": None,
            "snapshot_asks": None,
        })

    events = pd.DataFrame(rows)
    return events, snapshots

clean_events, clean_snapshots = simulate_clean_event_stream(config)

clean_events.head(), clean_events["event_type"].value_counts()

## 8. Corrupt the feed

Real feeds can have issues:

- duplicate messages;
- dropped messages;
- out-of-order messages;
- snapshot gaps;
- stale updates.

We introduce controlled corruption to test reconstruction.

In [ ]:
def corrupt_event_stream(clean_events, config):
    rng = np.random.default_rng(config.seed + 200)
    rows = []

    for _, row in clean_events.iterrows():
        if row["event_type"] != "SNAPSHOT" and rng.uniform() < config.corruption_drop_prob:
            continue

        rows.append(row.to_dict())

        if row["event_type"] != "SNAPSHOT" and rng.uniform() < config.corruption_duplicate_prob:
            duplicate = row.to_dict()
            rows.append(duplicate)

    corrupted = pd.DataFrame(rows)

    # Local shuffling within small windows to simulate out-of-order arrival.
    chunks = []
    for start in range(0, len(corrupted), config.corruption_shuffle_window):
        chunk = corrupted.iloc[start:start + config.corruption_shuffle_window].copy()
        if len(chunk) > 1 and rng.uniform() < 0.20:
            chunk = chunk.sample(frac=1.0, random_state=int(rng.integers(0, 1_000_000)))
        chunks.append(chunk)

    corrupted = pd.concat(chunks, ignore_index=True)
    corrupted["arrival_order"] = np.arange(len(corrupted))

    return corrupted

raw_events = corrupt_event_stream(clean_events, config)

raw_events.head(), len(clean_events), len(raw_events)

## 9. Feed integrity diagnostics

Before reconstructing, inspect:

- duplicate sequence numbers;
- missing sequence numbers;
- out-of-order arrivals;
- snapshot availability.

In [ ]:
def feed_integrity_report(raw_events):
    seq = raw_events["seq"].astype(int)

    duplicate_count = int(seq.duplicated().sum())
    unique_seq = np.sort(seq.unique())
    expected = np.arange(unique_seq.min(), unique_seq.max() + 1)
    missing = np.setdiff1d(expected, unique_seq)
    out_of_order_count = int((seq.diff().fillna(1) < 0).sum())
    snapshot_count = int((raw_events["event_type"] == "SNAPSHOT").sum())

    report = pd.DataFrame([{
        "n_messages": len(raw_events),
        "min_seq": int(seq.min()),
        "max_seq": int(seq.max()),
        "unique_seq_count": int(len(unique_seq)),
        "duplicate_seq_count": duplicate_count,
        "missing_seq_count": int(len(missing)),
        "out_of_order_adjacent_count": out_of_order_count,
        "snapshot_count": snapshot_count,
        "first_missing_seq": int(missing[0]) if len(missing) else -1,
        "last_missing_seq": int(missing[-1]) if len(missing) else -1,
    }])

    missing_df = pd.DataFrame({"missing_seq": missing})
    duplicate_df = raw_events[raw_events["seq"].duplicated(keep=False)].sort_values("seq")

    return report, missing_df, duplicate_df

integrity_report, missing_seq, duplicate_events = feed_integrity_report(raw_events)

integrity_report, missing_seq.head(), duplicate_events.head()

## 10. Reconstruct book with gap recovery

Reconstruction logic:

1. sort by sequence number;
2. drop duplicate sequence numbers;
3. start from first snapshot;
4. apply incremental updates;
5. if a sequence gap is detected:
   - mark gap;
   - stop applying increments until next snapshot;
   - recover from snapshot;
6. validate checksum when available.

This is conservative. In production, small gaps may be repaired from retransmission channels.

In [ ]:
def reconstruct_book_from_events(raw_events, n_levels, config):
    # Sort by sequence to reconstruct feed order.
    events_sorted = raw_events.sort_values(["seq", "arrival_order"]).copy()

    # Drop duplicates by sequence, keep first arrival.
    events_sorted["is_duplicate_seq"] = events_sorted["seq"].duplicated(keep="first")
    events_dedup = events_sorted[~events_sorted["is_duplicate_seq"]].copy()

    bids = {}
    asks = {}
    active = False
    expected_next_seq = None

    book_rows = []
    anomaly_rows = []
    gap_active = False

    for _, row in events_dedup.iterrows():
        seq = int(row["seq"])
        event_type = row["event_type"]

        if expected_next_seq is not None and seq != expected_next_seq:
            gap_size = seq - expected_next_seq
            anomaly_rows.append({
                "seq": seq,
                "anomaly_type": "SEQUENCE_GAP",
                "details": f"expected {expected_next_seq}, got {seq}",
                "gap_size": gap_size,
            })
            gap_active = True
            active = False

        if event_type == "SNAPSHOT":
            bids = dict(row["snapshot_bids"])
            asks = dict(row["snapshot_asks"])
            active = True
            gap_active = False
            expected_next_seq = seq + 1
        else:
            if not active:
                anomaly_rows.append({
                    "seq": seq,
                    "anomaly_type": "SKIPPED_DURING_GAP",
                    "details": "increment skipped until next snapshot",
                    "gap_size": np.nan,
                })
                expected_next_seq = seq + 1
                continue

            event = {
                "event_type": event_type,
                "side": row["side"],
                "price": row["price"],
                "size": row["size"],
            }
            try:
                bids, asks = apply_event_to_book(bids, asks, event)
            except Exception as exc:
                anomaly_rows.append({
                    "seq": seq,
                    "anomaly_type": "APPLY_ERROR",
                    "details": repr(exc),
                    "gap_size": np.nan,
                })

            expected_next_seq = seq + 1

        bids = clean_book_side(bids)
        asks = clean_book_side(asks)

        best_bid, best_ask, bid_size, ask_size = best_bid_ask(bids, asks)
        checksum_recalc = simple_book_checksum(bids, asks, n_levels=5)
        checksum_feed = int(row["checksum"]) if pd.notna(row["checksum"]) else -1
        checksum_ok = checksum_recalc == checksum_feed

        if not checksum_ok:
            anomaly_rows.append({
                "seq": seq,
                "anomaly_type": "CHECKSUM_MISMATCH",
                "details": f"feed {checksum_feed}, reconstructed {checksum_recalc}",
                "gap_size": np.nan,
            })

        crossed = bool(pd.notna(best_bid) and pd.notna(best_ask) and best_bid >= best_ask)
        if crossed:
            anomaly_rows.append({
                "seq": seq,
                "anomaly_type": "CROSSED_BOOK",
                "details": f"best_bid {best_bid}, best_ask {best_ask}",
                "gap_size": np.nan,
            })

        levels = book_to_levels(bids, asks, n_levels)
        book_rows.append({
            "seq": seq,
            "event_time_ns": row["event_time_ns"],
            "event_type": event_type,
            "best_bid": best_bid,
            "best_ask": best_ask,
            "bid_size": bid_size,
            "ask_size": ask_size,
            "spread": best_ask - best_bid if pd.notna(best_bid) and pd.notna(best_ask) else np.nan,
            "mid": (best_bid + best_ask) / 2 if pd.notna(best_bid) and pd.notna(best_ask) else np.nan,
            "checksum_feed": checksum_feed,
            "checksum_recalc": checksum_recalc,
            "checksum_ok": checksum_ok,
            "crossed": crossed,
            "active": active,
            **levels,
        })

    book = pd.DataFrame(book_rows)
    anomalies = pd.DataFrame(anomaly_rows)

    return book, anomalies, events_dedup

reconstructed_book, anomalies, events_dedup = reconstruct_book_from_events(raw_events, config.n_levels, config)

reconstructed_book.head(), anomalies.head(), len(anomalies)

## 11. Reconstruction quality report

Summarise:

- number of reconstructed states;
- checksum pass rate;
- gap count;
- skipped increment count;
- crossed-book count;
- duplicate messages removed.

In [ ]:
def reconstruction_quality_report(raw_events, events_dedup, reconstructed_book, anomalies):
    anomaly_counts = anomalies["anomaly_type"].value_counts().to_dict() if len(anomalies) else {}

    report = pd.DataFrame([{
        "raw_message_count": len(raw_events),
        "dedup_message_count": len(events_dedup),
        "duplicates_removed": int(len(raw_events) - len(events_dedup)),
        "reconstructed_state_count": len(reconstructed_book),
        "checksum_pass_rate": reconstructed_book["checksum_ok"].mean() if len(reconstructed_book) else np.nan,
        "sequence_gap_count": int(anomaly_counts.get("SEQUENCE_GAP", 0)),
        "skipped_during_gap_count": int(anomaly_counts.get("SKIPPED_DURING_GAP", 0)),
        "checksum_mismatch_count": int(anomaly_counts.get("CHECKSUM_MISMATCH", 0)),
        "crossed_book_count": int(anomaly_counts.get("CROSSED_BOOK", 0)),
        "apply_error_count": int(anomaly_counts.get("APPLY_ERROR", 0)),
        "active_state_rate": reconstructed_book["active"].mean() if len(reconstructed_book) else np.nan,
    }])

    return report

quality_report = reconstruction_quality_report(raw_events, events_dedup, reconstructed_book, anomalies)

quality_report.T

## 12. Reconstruct clean feed for baseline comparison

We also reconstruct the clean feed.

This gives a baseline for how corruption affects reconstruction quality.

In [ ]:
clean_raw = clean_events.copy()
clean_raw["arrival_order"] = np.arange(len(clean_raw))
clean_book, clean_anomalies, clean_dedup = reconstruct_book_from_events(clean_raw, config.n_levels, config)
clean_quality_report = reconstruction_quality_report(clean_raw, clean_dedup, clean_book, clean_anomalies)

clean_quality_report.T

## 13. L1 feature extraction

From reconstructed book states, compute:

- mid;
- spread;
- relative spread;
- best bid/ask sizes;
- top-of-book imbalance;
- microprice.

Microprice:

$$
microprice = \frac{ask \cdot bidSize + bid \cdot askSize} {bidSize+askSize}
$$

If bid size dominates, microprice moves toward ask, indicating upward pressure.

In [ ]:
def compute_l1_features(book):
    df = book.copy()

    df["relative_spread"] = df["spread"] / df["mid"]
    df["top_imbalance"] = (df["bid_size"] - df["ask_size"]) / (df["bid_size"] + df["ask_size"])
    df["microprice"] = (
        df["best_ask"] * df["bid_size"] + df["best_bid"] * df["ask_size"]
    ) / (df["bid_size"] + df["ask_size"])
    df["microprice_minus_mid"] = df["microprice"] - df["mid"]
    df["microprice_basis_points"] = df["microprice_minus_mid"] / df["mid"] * 10000.0
    df["mid_return_next"] = df["mid"].pct_change().shift(-1)
    df["seq_gap"] = df["seq"].diff().fillna(1) != 1

    return df

l1_features = compute_l1_features(reconstructed_book)
clean_l1_features = compute_l1_features(clean_book)

l1_features.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(l1_features["seq"], l1_features["mid"], label="corrupted/recovered")
plt.plot(clean_l1_features["seq"], clean_l1_features["mid"], label="clean", alpha=0.55)
plt.title("Reconstructed Mid Price")
plt.xlabel("Sequence")
plt.ylabel("Mid")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(l1_features["seq"], l1_features["top_imbalance"], alpha=0.8)
plt.axhline(0, linestyle="--")
plt.title("Top-of-Book Imbalance")
plt.xlabel("Sequence")
plt.ylabel("Imbalance")
plt.show()

## 14. L2 depth features

For $K$ levels:

$$
BidDepth_K = \sum_{i=1}^{K} bidSize_i
$$

$$
AskDepth_K = \sum_{i=1}^{K} askSize_i
$$

$$
DepthImbalance_K = \frac{BidDepth_K-AskDepth_K}{BidDepth_K+AskDepth_K}
$$

In [ ]:
def compute_l2_features(book, levels=(1, 3, 5, 10)):
    df = book[["seq", "event_time_ns", "mid", "spread"]].copy()

    for k in levels:
        bid_cols = [f"bid_sz_{i}" for i in range(1, k + 1) if f"bid_sz_{i}" in book.columns]
        ask_cols = [f"ask_sz_{i}" for i in range(1, k + 1) if f"ask_sz_{i}" in book.columns]

        bid_depth = book[bid_cols].sum(axis=1)
        ask_depth = book[ask_cols].sum(axis=1)

        df[f"bid_depth_{k}"] = bid_depth
        df[f"ask_depth_{k}"] = ask_depth
        df[f"depth_imbalance_{k}"] = (bid_depth - ask_depth) / (bid_depth + ask_depth).replace(0, np.nan)

    return df

l2_features = compute_l2_features(reconstructed_book)

l2_features.head()

In [ ]:
plt.figure(figsize=(12, 5))
for k in [1, 3, 5, 10]:
    col = f"depth_imbalance_{k}"
    if col in l2_features:
        plt.plot(l2_features["seq"], l2_features[col], label=col, alpha=0.8)
plt.axhline(0, linestyle="--")
plt.title("Depth Imbalance by Level")
plt.xlabel("Sequence")
plt.ylabel("Depth imbalance")
plt.legend()
plt.show()

## 15. Trade-through checks

A trade-through can indicate data or reconstruction problems.

For an aggressive buy trade, execution should not occur below the best ask.

For an aggressive sell trade, execution should not occur above the best bid.

In this simplified feed, the `TRADE` event price should be at the resting side level.

In [ ]:
def trade_through_checks(events_dedup, reconstructed_book):
    trades = events_dedup[events_dedup["event_type"] == "TRADE"].copy()
    book_ref = reconstructed_book[["seq", "best_bid", "best_ask", "mid"]].copy()

    trades = trades.merge(book_ref, on="seq", how="left")

    # Since book is recorded after applying the trade, allow one tick tolerance.
    trades["trade_through"] = False

    buy_mask = trades["aggressor_side"] == "BUY"
    sell_mask = trades["aggressor_side"] == "SELL"

    trades.loc[buy_mask, "trade_through"] = trades.loc[buy_mask, "price"] > trades.loc[buy_mask, "best_ask"] + config.tick_size
    trades.loc[sell_mask, "trade_through"] = trades.loc[sell_mask, "price"] < trades.loc[sell_mask, "best_bid"] - config.tick_size

    report = pd.DataFrame([{
        "n_trades": len(trades),
        "trade_through_count": int(trades["trade_through"].sum()),
        "trade_through_rate": trades["trade_through"].mean() if len(trades) else np.nan,
    }])

    return trades, report

trade_checks, trade_through_report = trade_through_checks(events_dedup, reconstructed_book)

trade_through_report

## 16. Stale quote detection

If the best bid and ask do not change for too many events, the quote stream may be stale.

This can happen due to:

- feed interruption;
- reconstruction stuck after gap;
- replay problem;
- exchange halt.

We compute runs of unchanged top-of-book.

In [ ]:
def stale_quote_detection(book, threshold_events):
    df = book[["seq", "best_bid", "best_ask"]].copy()
    changed = (df["best_bid"].diff() != 0) | (df["best_ask"].diff() != 0)
    group = changed.cumsum()
    df["unchanged_run_length"] = df.groupby(group).cumcount() + 1
    df["stale_quote_flag"] = df["unchanged_run_length"] > threshold_events

    report = pd.DataFrame([{
        "max_unchanged_run_length": int(df["unchanged_run_length"].max()),
        "stale_event_count": int(df["stale_quote_flag"].sum()),
        "stale_event_rate": df["stale_quote_flag"].mean(),
    }])

    return df, report

stale_quotes, stale_quote_report = stale_quote_detection(reconstructed_book, config.stale_quote_event_threshold)

stale_quote_report

## 17. Checksum mismatch analysis

A checksum mismatch means reconstructed state disagrees with the feed’s expected state.

Common reasons:

- dropped increment;
- duplicate incorrectly applied;
- out-of-order update;
- wrong snapshot;
- different aggregation convention;
- feed bug.

This notebook intentionally corrupts messages, so mismatches are expected after gaps until recovery.

In [ ]:
checksum_mismatches = reconstructed_book[~reconstructed_book["checksum_ok"]].copy()

checksum_mismatch_report = pd.DataFrame([{
    "checksum_mismatch_count": len(checksum_mismatches),
    "checksum_mismatch_rate": len(checksum_mismatches) / max(len(reconstructed_book), 1),
    "first_mismatch_seq": int(checksum_mismatches["seq"].iloc[0]) if len(checksum_mismatches) else -1,
    "last_mismatch_seq": int(checksum_mismatches["seq"].iloc[-1]) if len(checksum_mismatches) else -1,
}])

checksum_mismatch_report

## 18. Feature predictive sanity check

Order book features should have at least plausible relationships with future mid-price moves.

We test simple correlations:

- top imbalance;
- microprice minus mid;
- depth imbalance.

This is not alpha research. It is a reconstruction sanity check.

In [ ]:
def feature_sanity_report(l1_features, l2_features):
    df = l1_features[["seq", "top_imbalance", "microprice_basis_points", "mid_return_next"]].merge(
        l2_features[["seq", "depth_imbalance_3", "depth_imbalance_5", "depth_imbalance_10"]],
        on="seq",
        how="left",
    ).dropna()

    rows = []
    for feature in ["top_imbalance", "microprice_basis_points", "depth_imbalance_3", "depth_imbalance_5", "depth_imbalance_10"]:
        rows.append({
            "feature": feature,
            "corr_with_next_mid_return": df[feature].corr(df["mid_return_next"]),
            "mean": df[feature].mean(),
            "std": df[feature].std(),
        })

    return pd.DataFrame(rows)

sanity_report = feature_sanity_report(l1_features, l2_features)

sanity_report

## 19. Replay benchmark

Replay speed matters when reconstructing large feeds.

We benchmark reconstruction on the raw event stream.

In [ ]:
def benchmark_reconstruction(raw_events, config):
    rows = []

    for repeat in range(config.benchmark_repeats):
        start = time.perf_counter()
        book, anom, dedup = reconstruct_book_from_events(raw_events, config.n_levels, config)
        elapsed = time.perf_counter() - start

        rows.append({
            "repeat": repeat,
            "elapsed_seconds": elapsed,
            "raw_messages": len(raw_events),
            "states_reconstructed": len(book),
            "raw_messages_per_second": len(raw_events) / elapsed,
            "states_per_second": len(book) / elapsed,
            "anomaly_count": len(anom),
        })

    return pd.DataFrame(rows)

benchmark_report = benchmark_reconstruction(raw_events, config)

benchmark_summary = pd.DataFrame([{
    "mean_elapsed_seconds": benchmark_report["elapsed_seconds"].mean(),
    "mean_raw_messages_per_second": benchmark_report["raw_messages_per_second"].mean(),
    "mean_states_per_second": benchmark_report["states_per_second"].mean(),
    "mean_anomaly_count": benchmark_report["anomaly_count"].mean(),
}])

benchmark_summary

## 20. Snapshot recovery diagnostics

Measure how many gaps were recovered by snapshots.

A conservative replay stops applying increments during a gap and resumes when a snapshot arrives.

In [ ]:
def snapshot_recovery_report(anomalies, reconstructed_book):
    gaps = anomalies[anomalies["anomaly_type"] == "SEQUENCE_GAP"].copy() if len(anomalies) else pd.DataFrame()
    skipped = anomalies[anomalies["anomaly_type"] == "SKIPPED_DURING_GAP"].copy() if len(anomalies) else pd.DataFrame()

    snapshots_after_gap = []
    if len(gaps):
        snapshot_seqs = reconstructed_book.loc[reconstructed_book["event_type"] == "SNAPSHOT", "seq"].to_numpy()
        for _, gap in gaps.iterrows():
            seq = gap["seq"]
            next_snapshots = snapshot_seqs[snapshot_seqs >= seq]
            snapshots_after_gap.append(int(next_snapshots[0]) if len(next_snapshots) else -1)

    report = pd.DataFrame([{
        "sequence_gap_count": len(gaps),
        "skipped_increment_count": len(skipped),
        "gaps_with_future_snapshot": int(sum(x >= 0 for x in snapshots_after_gap)),
        "mean_recovery_distance_seq": float(np.mean([x - gaps.iloc[i]["seq"] for i, x in enumerate(snapshots_after_gap) if x >= 0])) if len(snapshots_after_gap) else np.nan,
        "max_recovery_distance_seq": float(np.max([x - gaps.iloc[i]["seq"] for i, x in enumerate(snapshots_after_gap) if x >= 0])) if len(snapshots_after_gap) and any(x >= 0 for x in snapshots_after_gap) else np.nan,
    }])

    recovery_detail = pd.DataFrame({
        "gap_seq": gaps["seq"].to_numpy() if len(gaps) else [],
        "next_snapshot_seq": snapshots_after_gap,
    })

    if len(recovery_detail):
        recovery_detail["recovery_distance"] = recovery_detail["next_snapshot_seq"] - recovery_detail["gap_seq"]

    return report, recovery_detail

recovery_report, recovery_detail = snapshot_recovery_report(anomalies, reconstructed_book)

recovery_report, recovery_detail.head()

## 21. Reconciliation against clean book

Because this is synthetic, we have the clean feed.

We compare reconstructed corrupted/recovered book against the clean reconstruction by sequence number.

In [ ]:
def compare_to_clean_book(reconstructed_book, clean_book):
    cols = ["seq", "best_bid", "best_ask", "bid_size", "ask_size", "mid", "spread"]
    merged = reconstructed_book[cols].merge(
        clean_book[cols],
        on="seq",
        how="inner",
        suffixes=("_recon", "_clean"),
    )

    for col in ["best_bid", "best_ask", "bid_size", "ask_size", "mid", "spread"]:
        merged[f"{col}_abs_diff"] = (merged[f"{col}_recon"] - merged[f"{col}_clean"]).abs()

    report = pd.DataFrame([{
        "matched_seq_count": len(merged),
        "mid_exact_match_rate": (merged["mid_abs_diff"] < 1e-12).mean(),
        "best_bid_exact_match_rate": (merged["best_bid_abs_diff"] < 1e-12).mean(),
        "best_ask_exact_match_rate": (merged["best_ask_abs_diff"] < 1e-12).mean(),
        "mean_mid_abs_diff": merged["mid_abs_diff"].mean(),
        "p95_mid_abs_diff": merged["mid_abs_diff"].quantile(0.95),
        "max_mid_abs_diff": merged["mid_abs_diff"].max(),
    }])

    return merged, report

clean_comparison, clean_comparison_report = compare_to_clean_book(reconstructed_book, clean_book)

clean_comparison_report.T

## 22. Reconstructed dataset assembly

The final data product combines:

- L1 features;
- L2 features;
- feed integrity flags;
- anomaly markers;
- trade markers.

This is the table downstream models should consume.

In [ ]:
def assemble_reconstructed_dataset(l1_features, l2_features, anomalies, events_dedup):
    dataset = l1_features.merge(
        l2_features.drop(columns=["event_time_ns", "mid", "spread"], errors="ignore"),
        on="seq",
        how="left",
    )

    anomaly_flags = pd.crosstab(anomalies["seq"], anomalies["anomaly_type"]) if len(anomalies) else pd.DataFrame()
    if len(anomaly_flags):
        anomaly_flags = anomaly_flags.reset_index()
        dataset = dataset.merge(anomaly_flags, on="seq", how="left")

    for col in ["SEQUENCE_GAP", "SKIPPED_DURING_GAP", "CHECKSUM_MISMATCH", "CROSSED_BOOK", "APPLY_ERROR"]:
        if col not in dataset.columns:
            dataset[col] = 0
        dataset[col] = dataset[col].fillna(0).astype(int)

    trade_flags = events_dedup[["seq", "event_type", "side", "price", "size", "aggressor_side"]].copy()
    trade_flags["is_trade"] = (trade_flags["event_type"] == "TRADE").astype(int)
    dataset = dataset.merge(
        trade_flags[["seq", "is_trade", "aggressor_side"]],
        on="seq",
        how="left",
    )

    dataset["is_trade"] = dataset["is_trade"].fillna(0).astype(int)
    dataset["aggressor_side"] = dataset["aggressor_side"].fillna("")

    return dataset

reconstructed_dataset = assemble_reconstructed_dataset(l1_features, l2_features, anomalies, events_dedup)

reconstructed_dataset.head()

## 23. Governance flags

A reconstructed order book should be reviewed if:

1. sequence gaps are present;
2. checksum mismatch rate is high;
3. crossed books occur;
4. stale quote runs are long;
5. trade-through rate is high;
6. reconstruction diverges materially from snapshots/clean reference;
7. replay speed is too slow;
8. model features are produced from inactive/gap states.

In [ ]:
selected_quality = quality_report.iloc[0]
selected_trade = trade_through_report.iloc[0]
selected_stale = stale_quote_report.iloc[0]
selected_recovery = recovery_report.iloc[0]
selected_clean = clean_comparison_report.iloc[0]
selected_benchmark = benchmark_summary.iloc[0]

governance_flags = pd.DataFrame([{
    "sequence_gap_count": selected_quality["sequence_gap_count"],
    "checksum_mismatch_count": selected_quality["checksum_mismatch_count"],
    "checksum_pass_rate": selected_quality["checksum_pass_rate"],
    "crossed_book_count": selected_quality["crossed_book_count"],
    "trade_through_rate": selected_trade["trade_through_rate"],
    "max_stale_run": selected_stale["max_unchanged_run_length"],
    "stale_event_rate": selected_stale["stale_event_rate"],
    "mid_exact_match_rate_vs_clean": selected_clean["mid_exact_match_rate"],
    "max_mid_abs_diff_vs_clean": selected_clean["max_mid_abs_diff"],
    "mean_states_per_second": selected_benchmark["mean_states_per_second"],
    "active_state_rate": selected_quality["active_state_rate"],
    "flag_sequence_gaps_present": bool(selected_quality["sequence_gap_count"] > 0),
    "flag_checksum_pass_rate_low": bool(selected_quality["checksum_pass_rate"] < 0.995),
    "flag_crossed_books_present": bool(selected_quality["crossed_book_count"] > 0),
    "flag_trade_through_rate_high": bool(selected_trade["trade_through_rate"] > 0.001 if pd.notna(selected_trade["trade_through_rate"]) else False),
    "flag_stale_quote_run_high": bool(selected_stale["max_unchanged_run_length"] > config.stale_quote_event_threshold),
    "flag_clean_reconciliation_poor": bool(selected_clean["mid_exact_match_rate"] < 0.98),
    "flag_replay_speed_low": bool(selected_benchmark["mean_states_per_second"] < 10_000),
    "flag_inactive_state_rate_high": bool(selected_quality["active_state_rate"] < 0.95),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_sequence_gaps_present",
        "flag_checksum_pass_rate_low",
        "flag_crossed_books_present",
        "flag_trade_through_rate_high",
        "flag_stale_quote_run_high",
        "flag_clean_reconciliation_poor",
        "flag_replay_speed_low",
        "flag_inactive_state_rate_high",
    ]
].any(axis=1)

governance_flags

## 24. Audit checks

Numerical and process checks:

1. clean feed has no missing sequences;
2. raw feed has arrival order;
3. reconstructed book has positive spreads when active;
4. L1 features are finite where active;
5. L2 features are finite where active;
6. anomaly table exists;
7. benchmark output is finite;
8. governance flags exist.

In [ ]:
audit_rows = []

clean_seq = clean_events["seq"].astype(int)
clean_no_missing = bool(len(np.setdiff1d(np.arange(clean_seq.min(), clean_seq.max() + 1), np.sort(clean_seq.unique()))) == 0)
audit_rows.append({
    "check": "clean_feed_no_missing_sequences",
    "value": clean_no_missing,
    "passed": clean_no_missing,
})

raw_has_arrival_order = bool("arrival_order" in raw_events.columns)
audit_rows.append({
    "check": "raw_feed_has_arrival_order",
    "value": raw_has_arrival_order,
    "passed": raw_has_arrival_order,
})

active_book = reconstructed_book[reconstructed_book["active"]]
positive_spread = bool((active_book["spread"].dropna() > 0).all())
audit_rows.append({
    "check": "active_book_positive_spread",
    "value": positive_spread,
    "passed": positive_spread,
})

l1_finite_cols = ["mid", "spread", "relative_spread", "top_imbalance", "microprice"]
l1_finite = bool(np.isfinite(l1_features.loc[l1_features["active"], l1_finite_cols].dropna().to_numpy()).all())
audit_rows.append({
    "check": "l1_features_finite_when_active",
    "value": l1_finite,
    "passed": l1_finite,
})

l2_cols = [c for c in l2_features.columns if "depth" in c or "imbalance" in c]
l2_finite = bool(np.isfinite(l2_features[l2_cols].dropna().to_numpy()).all())
audit_rows.append({
    "check": "l2_features_finite",
    "value": l2_finite,
    "passed": l2_finite,
})

anomaly_table_exists = anomalies is not None
audit_rows.append({
    "check": "anomaly_table_exists",
    "value": anomaly_table_exists,
    "passed": bool(anomaly_table_exists),
})

benchmark_finite = bool(np.isfinite(benchmark_summary.select_dtypes(include=[float, int]).to_numpy()).all())
audit_rows.append({
    "check": "benchmark_output_finite",
    "value": benchmark_finite,
    "passed": benchmark_finite,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 25. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

# Snapshot dictionaries are not CSV-friendly, so save event CSV without snapshot payload columns.
clean_events_csv = clean_events.drop(columns=["snapshot_bids", "snapshot_asks"], errors="ignore")
raw_events_csv = raw_events.drop(columns=["snapshot_bids", "snapshot_asks"], errors="ignore")
events_dedup_csv = events_dedup.drop(columns=["snapshot_bids", "snapshot_asks"], errors="ignore")

clean_events_csv.to_csv(output_dir / "clean_events.csv", index=False)
raw_events_csv.to_csv(output_dir / "raw_events_corrupted.csv", index=False)
events_dedup_csv.to_csv(output_dir / "events_deduplicated.csv", index=False)

reconstructed_book.to_csv(output_dir / "reconstructed_book.csv", index=False)
clean_book.to_csv(output_dir / "clean_reconstructed_book.csv", index=False)
anomalies.to_csv(output_dir / "anomalies.csv", index=False)
integrity_report.to_csv(output_dir / "feed_integrity_report.csv", index=False)
missing_seq.to_csv(output_dir / "missing_sequences.csv", index=False)
duplicate_events.drop(columns=["snapshot_bids", "snapshot_asks"], errors="ignore").to_csv(output_dir / "duplicate_events.csv", index=False)

quality_report.to_csv(output_dir / "reconstruction_quality_report.csv", index=False)
clean_quality_report.to_csv(output_dir / "clean_quality_report.csv", index=False)
l1_features.to_csv(output_dir / "l1_features.csv", index=False)
l2_features.to_csv(output_dir / "l2_features.csv", index=False)
trade_checks.to_csv(output_dir / "trade_checks.csv", index=False)
trade_through_report.to_csv(output_dir / "trade_through_report.csv", index=False)
stale_quotes.to_csv(output_dir / "stale_quote_detection.csv", index=False)
stale_quote_report.to_csv(output_dir / "stale_quote_report.csv", index=False)
checksum_mismatch_report.to_csv(output_dir / "checksum_mismatch_report.csv", index=False)
sanity_report.to_csv(output_dir / "feature_sanity_report.csv", index=False)
benchmark_report.to_csv(output_dir / "benchmark_report.csv", index=False)
benchmark_summary.to_csv(output_dir / "benchmark_summary.csv", index=False)
recovery_report.to_csv(output_dir / "snapshot_recovery_report.csv", index=False)
recovery_detail.to_csv(output_dir / "snapshot_recovery_detail.csv", index=False)
clean_comparison.to_csv(output_dir / "clean_comparison.csv", index=False)
clean_comparison_report.to_csv(output_dir / "clean_comparison_report.csv", index=False)
reconstructed_dataset.to_csv(output_dir / "reconstructed_dataset.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "order_book_reconstruction_from_events_outputs",
    "schema_version": "order_book_reconstruction_from_events_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "raw_message_count": int(len(raw_events)),
    "deduplicated_message_count": int(len(events_dedup)),
    "reconstructed_state_count": int(len(reconstructed_book)),
    "integrity_report": integrity_report.to_dict(orient="records"),
    "quality_report": quality_report.to_dict(orient="records"),
    "trade_through_report": trade_through_report.to_dict(orient="records"),
    "stale_quote_report": stale_quote_report.to_dict(orient="records"),
    "clean_comparison_report": clean_comparison_report.to_dict(orient="records"),
    "benchmark_summary": benchmark_summary.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook reconstructs a price-level order book from synthetic event messages.",
        "The event feed includes snapshots and incremental add/modify/cancel/trade/delete updates.",
        "The corrupted feed intentionally contains duplicates, dropped messages and local out-of-order arrivals.",
        "The reconstruction engine sorts by sequence, deduplicates, detects gaps and recovers from snapshots.",
        "L1 and L2 features include spread, imbalance, microprice and depth imbalance.",
        "Checks include sequence gaps, checksums, crossed books, stale quotes, trade-throughs and clean-feed reconciliation.",
        "Real production use should replace synthetic events with exchange/vendor event feeds and official checksum rules."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "reconstructed_dataset.csv", output_dir / "reconstruction_quality_report.csv", output_dir / "governance_flags.csv", manifest_path

## 26. Practical checklist for order book reconstruction

Before trusting reconstructed book data:

1. **Verify sequence numbers.**
2. **Remove duplicates.**
3. **Detect dropped messages.**
4. **Recover from snapshots.**
5. **Validate checksums.**
6. **Check crossed books.**
7. **Check trade-throughs.**
8. **Detect stale quotes.**
9. **Separate active and gap states.**
10. **Save reconstruction diagnostics with every dataset.**

## 27. Limitations

### Synthetic feed

The notebook uses a synthetic event feed. Real feeds have exchange-specific event types and recovery protocols.

### Price-level book

The reconstruction tracks aggregate size by price level, not individual queue position.

### Simplified checksum

The checksum is illustrative. Production feeds use official checksum algorithms.

### Conservative gap handling

The engine skips increments during gaps until the next snapshot. Real systems may use retransmission channels.

### Simplified trade-through rules

Real trade-through logic depends on venue rules, hidden liquidity, auctions, spreads, and event ordering.

### Memory and performance

The replay engine is pure Python. Production reconstruction should use C++/Rust or vectorised/event-batched approaches.

## 28. Research frontier and extensions

Important extensions include:

1. order-level reconstruction;
2. queue-position estimation;
3. official exchange checksum support;
4. retransmission channel simulation;
5. event-time versus receive-time analysis;
6. multi-venue book reconstruction;
7. nanosecond timestamp reconciliation;
8. C++ book replay engine;
9. L2 feature store generation;
10. integration with fill-probability and smart-router notebooks.

## 29. Phase 6 closing note

Phase 6 built the execution and microstructure systems layer:

1. Almgren-Chriss optimal execution;
2. dynamic programming execution;
3. square-root market impact;
4. Hawkes order flow;
5. rough volatility estimation;
6. microstructure friction C++ core;
7. limit-board, margin and deadband rebalancing;
8. Avellaneda-Stoikov market making;
9. limit-order fill probability;
10. smart order router;
11. C++/Python bindings with pybind11;
12. latency budget profiler;
13. order book reconstruction from events.

Together, these notebooks move the project from “research strategy” to “execution-aware quantitative system.”

## 30. Summary

This notebook implemented order book reconstruction from events.

It showed how to:

1. define snapshot and incremental event schemas;
2. simulate a clean exchange event feed;
3. intentionally corrupt the feed;
4. detect missing, duplicate and out-of-order messages;
5. reconstruct a price-level order book;
6. recover from snapshots after sequence gaps;
7. validate checksums;
8. detect crossed books;
9. extract L1 features;
10. extract L2 depth features;
11. check trade-throughs;
12. detect stale quotes;
13. benchmark replay speed;
14. compare corrupted reconstruction to clean baseline;
15. assemble downstream feature datasets;
16. create governance flags and audit checks;
17. save outputs and manifests;
18. close Phase 6.

The key computational takeaway:

> Order book reconstruction is a deterministic replay problem with strict sequence, checksum, and state-validity requirements.

The key financial takeaway:

> Execution models, fill models, and microstructure signals are only as good as the book states they consume.

## 31. Further reading

- Exchange market-data feed specifications.
- NASDAQ ITCH / OUCH documentation.
- CME MDP market data documentation.
- LOBSTER order book reconstruction methodology.
- Abergel et al., *Limit Order Books.*
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Harris, *Trading and Exchanges.*